# Data Preparation

This notebook acts as the foundational step for the telemetry analysis pipeline. It is responsible for loading the selected team and Grand Prix data, fetching both drivers' session information, and executing a robust cleaning and interpolation workflow. The resulting processed telemetry is then cached for efficient reuse across subsequent analytical and visualization notebooks.

##  Team and Grand Prix Selection

Configure the primary parameters for the analysis: the target team, specific Grand Prix, and racing season. These variables serve as the input for all subsequent data fetching functions, ensuring the analysis is strictly scoped to the desired racing event.

In [1]:
team = "McLaren"
grand_prix = "British Grand Prix"
year = 2025

## Setup and Dependencies


Import the necessary external libraries (such as fastf1) and initialize your internal project modules located in the src directory. This section also sets the project_root to ensure consistent file path resolution across the entire repository.

In [2]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_save_data import *
from src.preprocess_data import *

## Get Drivers Telemetry

Identify the two teammates for the selected team and session. This step retrieves the raw lap and telemetry datasets from the fastf1 API. The raw data is briefly inspected before being stored in the cache for preliminary record-keeping.

In [3]:
session = fastf1.get_session(year, grand_prix, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, team)
segment = get_segment(session, driver_1, driver_2)

try:
    lap_1, lap_2 = load_teammates_data(year, grand_prix, segment, driver_1, driver_2)

    print(f"\nDrivers names: {lap_1['Driver']}, {lap_2['Driver']}")
    print(f"Lap times: {lap_1['LapTime']}, {lap_2['LapTime']}")
    print(f"Lap start times: {lap_1['LapStartTime']}, {lap_2['LapStartTime']}")
    print(f"Tyre compounds: {lap_1['TyreCompound']}, {lap_2['TyreCompound']}")
    print(f"Tyre ages: {lap_1['TyreAge']}, {lap_2['TyreAge']}", "\n")
    print(f"{lap_1['Driver']} telemetry data info:\n")
    print(lap_1['Telemetry'].info(), "\n")
    print(f"{lap_1['Driver']} telemetry data head:\n", lap_1['Telemetry'].head(), "\n")
    print(f"{lap_2['Driver']} telemetry data info:\n")
    print(lap_2['Telemetry'].info(), "\n")
    print(f"{lap_2['Driver']} telemetry data head:\n", lap_2['Telemetry'].head(), "\n")

except Exception as e:
    print('Error loading telemetry data:', e)

req         WARNING 	DEFAULT CACHE ENABLED! (99.92 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for British Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '63', '44', '16', '12', '87', '14', '10', '55', '22', '6', '23', '31', '30', '5', '18', '27', '43']



Drivers names: PIA, NOR
Lap times: 84.995, 85.01
Lap start times: 0 days 01:15:20.045000, 0 days 01:23:17.659000
Tyre compounds: SOFT, SOFT
Tyre ages: 2, 2 

PIA telemetry data info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Distance  631 non-null    float64
 1   Speed     631 non-null    float64
 2   RPM       631 non-null    int64  
 3   Throttle  631 non-null    float64
 4   nGear     631 non-null    int64  
 5   Brake     631 non-null    int64  
 6   Time      631 non-null    float64
dtypes: float64(4), int64(3)
memory usage: 34.6 KB
None 

PIA telemetry data head:
    Distance    Speed    RPM  Throttle  nGear  Brake   Time
0     0.000  267.119  10653      99.0      7      0  0.000
1     9.243  269.520  10762      99.0      7      0  0.123
2    18.487  271.963  10872      99.0      7      0  0.245
3    27.730  273.493  10908      99.0   

## Data Preprocessing

Standardize and clean the raw telemetry data. This includes handling data inconsistencies, formatting timestamps, and ensuring that all telemetry channels are prepared for comparative analysis. The cleaned outputs are then cached, allowing future notebooks to bypass this computationally intensive step.

In [4]:
try:
    lap_1, lap_2 = preprocess_teammates_data(year, grand_prix, segment, lap_1, lap_2)

    print(f"Drivers names: {lap_1['Driver']}, {lap_2['Driver']}")
    print(f"Preprocessed lap times: {lap_1['LapTime']}, {lap_2['LapTime']}")
    print(f"Preprocessed lap start times: {lap_1['LapStartTime']}, {lap_2['LapStartTime']}")
    print(f"Preprocessed tyre compounds: {lap_1['TyreCompound']}, {lap_2['TyreCompound']}")
    print(f"Preprocessed tyre ages: {lap_1['TyreAge']}, {lap_2['TyreAge']}", "\n")
    print(f"{lap_1['Driver']} telemetry data info:\n")
    print(lap_1['Telemetry'].info(), "\n")
    print(f"{lap_1['Driver']} telemetry data head:\n", lap_1['Telemetry'].head(), "\n")
    print(f"{lap_2['Driver']} telemetry data info:\n")
    print(lap_2['Telemetry'].info(), "\n")
    print(f"{lap_2['Driver']} telemetry data head:\n", lap_2['Telemetry'].head(), "\n")

except Exception as e:
    print('Error preprocessing data:', e)
    

Drivers names: PIA, NOR
Preprocessed lap times: 84.995, 85.01
Preprocessed lap start times: 0 days 01:15:20.045000, 0 days 01:23:17.659000
Preprocessed tyre compounds: SOFT, SOFT
Preprocessed tyre ages: 2, 2 

PIA telemetry data info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Distance  631 non-null    float64
 1   Speed     631 non-null    float64
 2   RPM       631 non-null    int64  
 3   Throttle  631 non-null    float64
 4   nGear     631 non-null    int64  
 5   Brake     631 non-null    int64  
 6   Time      631 non-null    float64
dtypes: float64(4), int64(3)
memory usage: 34.6 KB
None 

PIA telemetry data head:
    Distance    Speed    RPM  Throttle  nGear  Brake   Time
0     0.000  267.119  10653      99.0      7      0  0.000
1     9.243  269.520  10762      99.0      7      0  0.123
2    18.487  271.963  10872      99.0      7    

## Interpolation

Resample the telemetry data from both drivers onto a unified distance axis. Since drivers cover different distances in slightly different timeframes, this normalization process is critical for enabling a direct, point-by-point performance comparison between teammates.


In [5]:
print(f"\n{lap_1['Driver']} distance vector: \n{lap_1['Telemetry']['Distance'].head()}")
print(f"\n{lap_2['Driver']} distance vector: \n{lap_2['Telemetry']['Distance'].head()}")

try:
    n_points = (len(lap_1['Telemetry']['Distance']) + len(lap_2['Telemetry']['Distance'])) // 2
    lap_1["Telemetry"], lap_2["Telemetry"] = interpolate_telemetry(lap_1['Telemetry'], lap_2['Telemetry'], lap_1['LapTime'], lap_2['LapTime'], n_points=n_points)
except Exception as e:
    print(f"Error interpolating telemetry data: {e}")
    
print(f"\n{lap_1['Driver']} interpolated data info:\n")
print(lap_1['Telemetry'].info())
print(f"\n{lap_2['Driver']} interpolated data info:\n")
print(lap_2['Telemetry'].info())

print(f"\n{lap_1['Driver']} interpolated data description:\n\n {lap_1['Telemetry'].describe()}")
print(f"\n{lap_2['Driver']} interpolated data description:\n\n {lap_2['Telemetry'].describe()}")

print(f"\n{lap_1['Driver']} interpolated data head:\n\n {lap_1['Telemetry'].head()}")
print(f"\n{lap_2['Driver']} interpolated data head:\n\n {lap_2['Telemetry'].head()}")


PIA distance vector: 
0     0.000
1     9.243
2    18.487
3    27.730
4    36.973
Name: Distance, dtype: float64

NOR distance vector: 
0     0.000
1     9.243
2    18.487
3    27.730
4    36.973
Name: Distance, dtype: float64

Interpolation check:
Total distance driver 1: 5823.290 m
Total distance driver 2: 5823.290 m
Difference: 0.000 m
Percentage difference: 0.000%
OK: distances near, alignment is a very close.

Time 1 (interpolated): 84.9950  |  LapTime 1: 84.9950
Time 2 (interpolated): 85.0100  |  LapTime 2: 85.0100
Delta (calculated): -0.0150
Delta (expected):   -0.0150
OK: final delta matches the official lap time gap.

PIA interpolated data info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Distance  631 non-null    float64
 1   Speed     631 non-null    float64
 2   RPM       631 non-null    int64  
 3   Throttle  631 non-null    float

## Cache Processed Data

Persist the fully prepared and interpolated datasets to local cache files. This workflow design ensures that downstream exploratory notebooks can immediately load the high-quality, pre-processed data without re-executing the entire retrieval and transformation pipeline.


In [6]:
try:
    save_cache(year, grand_prix, segment, lap_1, lap_2)
    save_dataframe(lap_1, "Telemetry", year, grand_prix)
    save_dataframe(lap_2, "Telemetry", year, grand_prix)
except Exception as e:
    print(f"Failed to save drivers data to cache: {e}")

Saving payload to E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_british_grand_prix_Q3_PIA.pkl
Saved: E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_british_grand_prix_Q3_PIA.pkl
Saving payload to E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_british_grand_prix_Q3_NOR.pkl
Saved: E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_british_grand_prix_Q3_NOR.pkl

Processed data for 2025 British Grand Prix saved to cache.
Driver 1: PIA, Driver 2: NOR, Segment: Q3
